In [2]:
# Notebook: sweep_inpaint_metrics.ipynb (single-cell version)
# Assumes:
#   data/images/1.png ... data/images/20.png
#   data/masks/1.png  ... data/masks/6.png
#
# Metrics (no GT needed):
#   - seam_grad_gap: |mean_grad_in - mean_grad_out|  on a boundary band
#   - seam_color_gap_lab: mean L2 color gap (Lab) between inner/outer boundary bands
#   - tv_band: total variation on the union band (lower is smoother)
#
# Runs:
#   baseline / repaint / bar_repaint with a param grid + multiple seeds
#
# Outputs:
#   outputs/sweep_<timestamp>/results.csv
#   outputs/sweep_<timestamp>/renders/<method>/<img>_<mask>/<seed>_<params>.png

import os, time, itertools, math
os.chdir("/workspace")
from dataclasses import asdict
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

import cv2
from scipy.ndimage import distance_transform_edt, binary_dilation, binary_erosion

from diffusers import StableDiffusionInpaintPipeline


# ---- your callbacks ----
from src.samplers.latent_resample import (
    RePaintLikeConfig, BARConfig,
    build_callback_repaint_like, build_callback_bar_repaint
)

# ----------------------------
# Paths / I/O
# ----------------------------
IMAGES_DIR = "data/images"
MASKS_DIR  = "data/masks"
OUT_ROOT   = "outputs"
os.makedirs(OUT_ROOT, exist_ok=True)

run_tag = time.strftime("sweep_%Y-%m-%d_%H-%M-%S")
OUT_DIR = os.path.join(OUT_ROOT, run_tag)
RENDERS_DIR = os.path.join(OUT_DIR, "renders")
os.makedirs(RENDERS_DIR, exist_ok=True)

# ----------------------------
# Model / inference settings
# ----------------------------
MODEL_ID = "sd2-community/stable-diffusion-2-inpainting"  # or your cfg.model.hf_model_id
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16  # try torch.bfloat16 if you want

# You can tune these:
PROMPT = "photorealistic, high quality"
NEG_PROMPT = ""
GUIDANCE = 7.5
NUM_STEPS = 30
WIDTH = 512
HEIGHT = 512

# Boundary band width in *image pixels*
BAND_PX = 5

# Seeds per setting (stability)
SEEDS = [0, 1, 2]

# ----------------------------
# Sweep grids
# ----------------------------
# Start small, then expand after you see trends.
BASELINE_GRID = [{}]

REPAINT_GRID = [
    {"jump_every": je, "p": p, "stop_jump_frac": sjf, "time_decay": True}
    for je in [5, 6, 8]
    for p  in [0.15, 0.2, 0.25]
    for sjf  in [0.7, 0.75]
]

BAR_GRID = [
    {"jump_every": je, "p_max": pm, "gamma": g, "rings": r, "stop_jump_frac": sjf, "time_decay": True}
    for je in [5, 6]
    for pm in [0.35, 0.40, 0.45]
    for g  in [1.5, 2.0]
    for r  in [1, 3]
    for sjf  in [0.7, 0.75]
]

METHODS = [
    ("baseline", BASELINE_GRID),
    ("repaint",  REPAINT_GRID),
    ("bar_repaint", BAR_GRID),
]

# ----------------------------
# Utils: load / resize
# ----------------------------
def load_image_rgb(path: str, size: Tuple[int,int]) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return img.resize(size, resample=Image.BICUBIC)

def load_mask_l_binary(path: str, size: Tuple[int,int]) -> Image.Image:
    m = Image.open(path).convert("L")
    m = m.resize(size, resample=Image.NEAREST)
    arr = np.array(m, dtype=np.uint8)
    arr = (arr >= 128).astype(np.uint8) * 255  # white=fill
    return Image.fromarray(arr, mode="L")

def pil_to_np_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))

def pil_to_np_mask01(mask_l: Image.Image) -> np.ndarray:
    arr = np.array(mask_l, dtype=np.uint8)
    return (arr >= 128).astype(np.uint8)  # 1=fill

# ----------------------------
# Boundary band construction
# ----------------------------
def boundary_bands(mask01: np.ndarray, band_px: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    mask01: [H,W] 1=fill, 0=keep
    Returns:
      band_in  : inside-fill pixels within band_px of boundary
      band_out : outside-fill pixels within band_px of boundary
      band_union: band_in OR band_out
    """
    mask = mask01.astype(bool)
    keep = ~mask

    # distance to opposite set
    # inside band: fill pixels close to keep
    d_in  = distance_transform_edt(mask)   # distance inside fill to nearest keep
    d_out = distance_transform_edt(keep)   # distance outside fill to nearest fill

    band_in  = mask & (d_in  <= band_px) & (d_in > 0)
    band_out = keep & (d_out <= band_px) & (d_out > 0)
    band_union = band_in | band_out
    return band_in, band_out, band_union

# ----------------------------
# Metrics (no GT)
# ----------------------------
def sobel_grad_mag(gray01: np.ndarray) -> np.ndarray:
    gx = cv2.Sobel(gray01, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray01, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx*gx + gy*gy)

def total_variation(img_rgb01: np.ndarray, mask_bool: np.ndarray) -> float:
    """
    TV on masked area only; img_rgb01 float32 in [0,1], mask_bool [H,W]
    """
    # compute per-channel finite diffs
    dx = np.abs(img_rgb01[:, 1:, :] - img_rgb01[:, :-1, :])
    dy = np.abs(img_rgb01[1:, :, :] - img_rgb01[:-1, :, :])

    # align masks
    m_dx = mask_bool[:, 1:] & mask_bool[:, :-1]
    m_dy = mask_bool[1:, :] & mask_bool[:-1, :]

    tv = dx[m_dx].mean() if m_dx.any() else 0.0
    tv += dy[m_dy].mean() if m_dy.any() else 0.0
    return float(tv)

def seam_metrics(out_rgb: np.ndarray, mask01: np.ndarray, band_px: int) -> Dict[str, float]:
    """
    out_rgb: [H,W,3] uint8
    mask01: [H,W] {0,1}
    """
    band_in, band_out, band_union = boundary_bands(mask01, band_px=band_px)

    img01 = out_rgb.astype(np.float32) / 255.0
    gray = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    grad = sobel_grad_mag(gray)

    # Seam gradient gap
    gin  = grad[band_in].mean()  if band_in.any()  else 0.0
    gout = grad[band_out].mean() if band_out.any() else 0.0
    seam_grad_gap = float(abs(gin - gout))

    # Color gap in Lab between bands (means)
    lab = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    if band_in.any() and band_out.any():
        mu_in  = lab[band_in].mean(axis=0)
        mu_out = lab[band_out].mean(axis=0)
        seam_color_gap_lab = float(np.linalg.norm(mu_in - mu_out))
    else:
        seam_color_gap_lab = 0.0

    # Total variation on union band (smoothness proxy; lower is better)
    tv_band = total_variation(img01, band_union)

    # Also useful: edge density difference (optional)
    edge = (grad > 0.15).astype(np.uint8)  # threshold in [0,1]
    ein  = edge[band_in].mean()  if band_in.any()  else 0.0
    eout = edge[band_out].mean() if band_out.any() else 0.0
    edge_density_gap = float(abs(ein - eout))

    return dict(
        seam_grad_gap=seam_grad_gap,
        seam_color_gap_lab=seam_color_gap_lab,
        tv_band=tv_band,
        edge_density_gap=edge_density_gap,
        band_in_px=int(band_in.sum()),
        band_out_px=int(band_out.sum()),
    )

# ----------------------------
# Inference runner
# ----------------------------
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

# Optional: compile for speed when doing many runs (enable when stable)
# pipe.unet = torch.compile(pipe.unet, mode="max-autotune", fullgraph=False)

def run_one(
    image_pil: Image.Image,
    mask_pil: Image.Image,
    method: str,
    params: Dict[str, Any],
    seed: int,
) -> Image.Image:
    set_seed(seed)

    callback = None
    if method == "repaint":
        rcfg = RePaintLikeConfig(
            jump_every=int(params["jump_every"]),
            p=float(params["p"]),
            stop_jump_frac=float(params.get("stop_jump_frac", 0.8)),
            time_decay=bool(params.get("time_decay", True)),
        )
        callback = build_callback_repaint_like(
            pipe=pipe, mask_l=mask_pil, cfg=rcfg,
            num_inference_steps=NUM_STEPS
        )

    elif method == "bar_repaint":
        rcfg = RePaintLikeConfig(
            jump_every=int(params["jump_every"]),
            p=0.0,
            stop_jump_frac=float(params.get("stop_jump_frac", 0.8)),
            time_decay=bool(params.get("time_decay", True)),
        )
        bcfg = BARConfig(
            p_max=float(params["p_max"]),
            gamma=float(params["gamma"]),
            rings=int(params["rings"]),
        )
        callback = build_callback_bar_repaint(
            pipe=pipe, mask_l=mask_pil,
            repaint_cfg=rcfg, bar_cfg=bcfg,
            num_inference_steps=NUM_STEPS
        )

    # baseline => callback None

    # AMP only on cuda + half
    use_amp = (DEVICE.type == "cuda" and DTYPE in (torch.float16, torch.bfloat16))
    amp_ctx = torch.autocast(device_type="cuda", dtype=DTYPE) if use_amp else torch.no_grad()

    with torch.inference_mode():
        if use_amp:
            ctx = torch.autocast(device_type="cuda", dtype=DTYPE)
        else:
            # dummy context
            class _Null:
                def __enter__(self): return None
                def __exit__(self, *a): return False
            ctx = _Null()

        with ctx:
            res = pipe(
                prompt=PROMPT,
                negative_prompt=NEG_PROMPT,
                image=image_pil,
                mask_image=mask_pil,
                guidance_scale=GUIDANCE,
                num_inference_steps=NUM_STEPS,
                callback_on_step_end=callback,
                callback_on_step_end_tensor_inputs=["latents"],
            )
    return res.images[0]

# ----------------------------
# Main sweep loop
# ----------------------------
# Image list: 1..20
image_ids = list(range(1, 21))
mask_ids  = list(range(1, 7))

rows = []

for img_id in tqdm(image_ids, desc="Images"):
    img_path = os.path.join(IMAGES_DIR, f"{img_id}.png")
    if not os.path.exists(img_path):
        print(f"[warn] missing image: {img_path}")
        continue

    image_pil = load_image_rgb(img_path, size=(WIDTH, HEIGHT))

    for m_id in mask_ids:
        mask_path = os.path.join(MASKS_DIR, f"{m_id}.png")
        if not os.path.exists(mask_path):
            print(f"[warn] missing mask: {mask_path}")
            continue

        mask_pil = load_mask_l_binary(mask_path, size=(WIDTH, HEIGHT))
        mask01 = pil_to_np_mask01(mask_pil)

        for method, grid in METHODS:
            for params in grid:
                # Render dir
                method_dir = os.path.join(RENDERS_DIR, method, f"img{img_id}_mask{m_id}")
                os.makedirs(method_dir, exist_ok=True)

                for seed in SEEDS:
                    out_pil = run_one(image_pil, mask_pil, method, params, seed)
                    out_np = pil_to_np_rgb(out_pil)

                    mets = seam_metrics(out_np, mask01, band_px=BAND_PX)

                    # Save image
                    # make compact filename
                    if method == "baseline":
                        fname = f"seed{seed}.png"
                    else:
                        ptag = "_".join([f"{k}{str(v).replace('.','p')}" for k,v in params.items()])
                        fname = f"seed{seed}__{ptag}.png"
                    out_file = os.path.join(method_dir, fname)
                    out_pil.save(out_file)

                    row = {
                        "img_id": img_id,
                        "mask_id": m_id,
                        "method": method,
                        "seed": seed,
                        "prompt": PROMPT,
                        "guidance": GUIDANCE,
                        "num_steps": NUM_STEPS,
                        "band_px": BAND_PX,
                        "out_file": out_file,
                        **(params if params else {}),
                        **mets,
                    }
                    rows.append(row)

df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "results.csv")
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

# Quick aggregate view: mean/std per method+params (helps pick optimal)
group_cols = ["method"]
# include params in grouping (only those present)
param_cols = sorted({c for c in df.columns if c in ["jump_every","p","p_max","gamma","rings","stop_jump_frac","time_decay"]})
group_cols += param_cols

agg = df.groupby(group_cols).agg(
    seam_grad_gap_mean=("seam_grad_gap","mean"),
    seam_grad_gap_std=("seam_grad_gap","std"),
    seam_color_gap_lab_mean=("seam_color_gap_lab","mean"),
    seam_color_gap_lab_std=("seam_color_gap_lab","std"),
    tv_band_mean=("tv_band","mean"),
    tv_band_std=("tv_band","std"),
).reset_index()

agg_path = os.path.join(OUT_DIR, "agg.csv")
agg.to_csv(agg_path, index=False)
print("Saved:", agg_path)

# Show top settings: prioritize low seam_grad_gap, then low color gap, then low tv
display(agg.sort_values(["seam_grad_gap_mean","seam_color_gap_lab_mean","tv_band_mean"]).head(20))


/opt/micromamba/envs/repaint-bar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-inpainting/snapshots/5f74973cbb64c8568780732c17f43eb269d63a0d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: outputs/sweep_2026-02-19_07-53-55/results.csv
Saved: outputs/sweep_2026-02-19_07-53-55/agg.csv


,method,gamma,jump_every,p,p_max,rings,stop_jump_frac,time_decay,seam_grad_gap_mean,seam_grad_gap_std,seam_color_gap_lab_mean,seam_color_gap_lab_std,tv_band_mean,tv_band_std


In [3]:
import os, glob
print("CWD:", os.getcwd())
print("exists data/images:", os.path.exists("data/images"))
print("exists data/masks :", os.path.exists("data/masks"))
print("images sample:", glob.glob("data/images/*")[:10])
print("masks sample :", glob.glob("data/masks/*")[:10])


CWD: /workspace/notebooks
exists data/images: False
exists data/masks : False
images sample: []
masks sample : []
